In [1]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/ECGPipes/notebooks
/Users/peli/Projects/Repositories/ECGPipes
Working Dir Base: /Users/peli/Projects/Repositories/ECGPipes


In [2]:
import yaml
import time
from nipype import Workflow, Node, MapNode, Function, IdentityInterface
from nipype.interfaces.io import SelectFiles, DataSink
# import
from src.preprocessing import *

loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


In [3]:
def create_meg_preprocessing(
    basedir: str,
    workdir: str,
    output_dir: str,
    subject_list: list[str],
    crop_params: dict,
    filter_params: dict,
    gradcomp_params: dict,
):
        
    wf = Workflow(name="megpreproc")
    # Create the nodes
    # create the subject_id "injector"?
    
    wf.base_dir = workdir
    subject_list = ['sub-01','sub-02']
    infosource = Node(IdentityInterface(fields=['subject_id']),
                    name="infosource")
    infosource.iterables = [('subject_id', subject_list)]

    templates = {"meg": "{subject_id}/meg/{subject_id}_task-MMNHCS_run-0_meg.fif"}
    # create the fileselector
    selectraw = Node(
        SelectFiles(templates, base_directory=basedir),
        name="selectfiles"
    )

    # Create the processing nodes

    # Cropping step
    crop = Node(Function(
        input_names=["in_file", "stim_channel", "min_buffer", "max_buffer"],
        output_names=["out_file"],
        function=crop_data,
        imports=["from src.preprocessing import *"]
    ), name='CropData')
    crop.inputs.stim_channel = crop_params["stim_channel"]
    crop.inputs.min_buffer = crop_params["min_buffer"]
    crop.inputs.max_buffer = crop_params["max_buffer"]

    # Rough filter step
    filter_node = Node(Function(
        input_names=["in_file", "l_freq", "h_freq"],
        output_names=["out_file"],
        function=filter_data
    ), name="FilterData")
    filter_node.inputs.l_freq = filter_params["l_freq"]
    filter_node.inputs.h_freq = filter_params["h_freq"]

    # Grad comp step
    grad_comp = Node(Function(
        input_names=["in_file", "auto", "order"],
        output_names=["out_file"],
        function=gradient_compensation
    ), name="GradientComp")
    grad_comp.inputs.auto = gradcomp_params["auto"]
    grad_comp.inputs.order = gradcomp_params["order"]

    # IO
    # Datasink - creates output folder for important outputs
    datasink = Node(DataSink(base_directory=output_dir,
                            container="datasink"),
                    name="datasink")

    wf.connect([
        (infosource, selectraw, [("subject_id", "subject_id")]),
        (selectraw, crop, [("meg", "in_file")]),
        (crop, filter_node, [("out_file", "in_file")]),
        (filter_node, grad_comp, [("out_file", "in_file")]),
        (grad_comp, datasink, [("out_file", "megpreproc.@final")]),
    ])

    return wf



In [4]:

# load configs
config_path = "config/config_ds006629.yaml"
with open(config_path, "r") as yamlfile:
    config = yaml.load(yamlfile, Loader=yaml.FullLoader)

paths_config = config["paths"]
wf_config = config["workflow"]
proc_config = config["processing"]

# set logger
from nipype import config, logging
config.update_config({'logging': {'log_directory': os.getcwd(),
                                  'log_to_file': True}})
logging.update_logging(config)

# create workflow
wf = create_meg_preprocessing(
    basedir=paths_config.get("basedir"),
    workdir=paths_config.get("workdir"),
    output_dir=paths_config.get("outputdir"),
    subject_list=paths_config.get("subjects"),
    crop_params=proc_config.get("crop"),
    filter_params=proc_config.get("filter"),
    gradcomp_params=proc_config.get("gradcomp")
)

# Run workflow 
# set nWorkers (cpu cores used first)
if wf_config["auto_workers"]:
    cpu_count = os.cpu_count()
    print(f"CPU cores available: {cpu_count}")
    n_workers = cpu_count - 2
    print(f"Set auto n_workers to {n_workers} workers")
else:
    n_workers = wf_config["n_workers"]

# visualise workflow

print(f"Running Workflow")
start = time.time()

wf.run(
    plugin=wf_config["plugin"], 
    plugin_args={
        "n_procs": n_workers
    })

end = time.time()
length = end - start
print(f"Finished workflow. Took {length} seconds.")

CPU cores available: 12
Set auto n_workers to 10 workers
Running Workflow
260224-11:49:40,141 nipype.workflow INFO:
	 Workflow megpreproc settings: ['check', 'execution', 'logging', 'monitoring']
260224-11:49:40,145 nipype.workflow INFO:
	 Running serially.
260224-11:49:40,145 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.selectfiles" in "/Users/peli/Projects/Repositories/ECGPipes/workdir/megpreproc/_subject_id_sub-01/selectfiles".
260224-11:49:40,147 nipype.workflow INFO:
	 [Node] Executing "selectfiles" <nipype.interfaces.io.SelectFiles>
260224-11:49:40,148 nipype.workflow INFO:
	 [Node] Finished "selectfiles", elapsed time 0.000136s.
260224-11:49:40,149 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.selectfiles" in "/Users/peli/Projects/Repositories/ECGPipes/workdir/megpreproc/_subject_id_sub-02/selectfiles".
260224-11:49:40,151 nipype.workflow INFO:
	 [Node] Executing "selectfiles" <nipype.interfaces.io.SelectFiles>
260224-11:49:40,151 nipype.workflow INFO:
	 [Node] 

RuntimeError: 2 raised. Re-raising first.

In [ ]:
# Write graph of type colored
wf.write_graph(graph2use='colored', dotfilename='./graph_colored.dot')

# Visualize graph
from IPython.display import Image
Image(filename="megpreproc/graph_colored.png")

OSError: No command "dot" found on host Elisiuss-MacBook-Pro.local. Please check that the corresponding package is installed.